<a href="https://colab.research.google.com/github/Aritra-221B/Aritras-Colab/blob/main/UnderstandingGenai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import google.generativeai as genai
from google.colab import userdata

# To use Gemini, you'll need an API key from Google AI Studio.
# Once you have it, add it to your Colab secrets as 'GOOGLE_API_KEY'.
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    print("API Key configured successfully.")
except Exception as e:
    print(f"Error: {e}. Please ensure GOOGLE_API_KEY is set in Colab secrets.")

# Placeholder for the model setup
# We will define the generation_config and the prompt in the next step.

API Key configured successfully.


### Understanding Generation Configuration

These parameters allow you to 'tune' the behavior of the model without retraining it:

*   **Temperature**: Controls randomness.
    *   **High (e.g., 0.9 - 1.0):** The model takes more risks, resulting in more creative, diverse, but potentially 'hallucinated' output.
    *   **Low (e.g., 0.1 - 0.2):** The model becomes deterministic, choosing the most likely words. Ideal for factual or structured tasks.
*   **Top-P (Nucleus Sampling)**: The model considers the smallest set of tokens whose cumulative probability sum is at least `P`. A lower `P` makes the output more focused.
*   **Top-K**: The model only samples from the top `K` most likely next words. This limits the vocabulary used in any given step.
*   **Max Output Tokens**: This is your 'budget.' It prevents the model from generating indefinitely, which is useful for cost control and ensuring concise responses.

In [40]:
# Using an available model confirmed by the list_models output
generation_config = {
    "temperature": 0.9,
    "top_p": 1,
    "top_k": 1,
    "max_output_tokens": 2048,
}

model = genai.GenerativeModel(
    model_name="models/gemini-2.5-flash",
    generation_config=generation_config,
)

print("Model initialized with gemini-2.5-flash")

Model initialized with gemini-2.5-flash


In [30]:
# Test the impact of max_output_tokens with the confirmed model

prompt = "Write a very detailed, long-winded description of a sunset over the ocean."

try:
    response = model.generate_content(prompt)
    print(f"Output Limit: {generation_config['max_output_tokens']} tokens")
    print(f"Result: {response.text}")
except Exception as e:
    print(f"Error: {e}")

Output Limit: 2048 tokens
Result: The day, having stretched its languid limbs across the boundless blue, began its slow, deliberate retreat, heralded by the first subtle whispers of twilight. It commenced not with a dramatic flourish, but with an almost imperceptible softening of the ocean’s harsh, silvery glare, as if the very air itself had taken a deep, calming breath. The sun, hitherto a searing, brilliant disc in the zenith, started its stately procession towards the western horizon, losing its blinding intensity with each passing moment, transforming into a benevolent, golden orb, magnified by the atmosphere into a colossal, burning coin poised precariously on the edge of the world.

As it dipped lower, the sky, a vast, unfathomable canvas, began its slow, deliberate metamorphosis. The fierce, crystalline azure of midday near the horizon yielded first to a tender, diluted wash of pale peach, like pigment dissolved in an endless expanse of water. This initial blush, delicate as th

In [32]:
# Let's explore the metadata of available models
# This helps you understand 'input_token_limit' and 'supported_generation_methods'

print("Available models and their capabilities:\n")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(f"Model Name: {m.name}")
        print(f"- Input Token Limit: {m.input_token_limit}")
        print(f"- Output Token Limit: {m.output_token_limit}")
        print(f"- Description: {m.description}\n")

Available models and their capabilities:

Model Name: models/gemini-2.5-flash
- Input Token Limit: 1048576
- Output Token Limit: 65536
- Description: Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.

Model Name: models/gemini-2.5-pro
- Input Token Limit: 1048576
- Output Token Limit: 65536
- Description: Stable release (June 17th, 2025) of Gemini 2.5 Pro

Model Name: models/gemini-2.0-flash
- Input Token Limit: 1048576
- Output Token Limit: 8192
- Description: Gemini 2.0 Flash

Model Name: models/gemini-2.0-flash-001
- Input Token Limit: 1048576
- Output Token Limit: 8192
- Description: Stable version of Gemini 2.0 Flash, our fast and versatile multimodal model for scaling across diverse tasks, released in January of 2025.

Model Name: models/gemini-2.0-flash-lite-001
- Input Token Limit: 1048576
- Output Token Limit: 8192
- Description: Stable version of Gemini 2.0 Flash-Lite

Model Name: models/gemini-2.

In [34]:
# Method 1: count_tokens
# This is useful for pre-calculating costs or ensuring you are within the context window.

sample_text = "The quick brown fox jumps over the lazy dog."

# You can count tokens for a specific model
token_count = model.count_tokens(sample_text)

print(f"Text: {sample_text}")
print(f"Total tokens: {token_count.total_tokens}")

Text: The quick brown fox jumps over the lazy dog.
Total tokens: 10


In [41]:
# Method 3: Chat Sessions
# This manages conversation history for you automatically.

chat = model.start_chat(history=[])

response = chat.send_message("In one sentence, who is the best person to describe a sunset?")
print(f"User: In one sentence, who is the best person to describe a sunset?")
print(f"Model: {response.text}\n")

# The model 'remembers' the previous turn
response = chat.send_message("Now, describe it in that person's voice.")
print(f"User: Now, describe it in that person's voice.")
print(f"Model: {response.text}")

User: In one sentence, who is the best person to describe a sunset?
Model: The person whose heart is most deeply stirred by its beauty and who possesses the words to convey that feeling.

User: Now, describe it in that person's voice.
Model: Ah, there it is... not just a sight, but a complete surrender. The sky, that vast, indifferent canvas, begins to bleed; molten gold first, then blushing into the most tender peach, before deepening through fiery tangerine and on into a bruised, hopeful lavender. Each cloud, a whisper-thin veil moments ago, now catches the light like a painter's desperate, joyful stroke, edged in impossible crimson and glowing from within.

The sun doesn't just descend; it *melts*, a grand, theatrical performance of bowing out, sinking slowly below the horizon as if dragging the last vestiges of the day's warmth with it. For a breath, the world holds still, suspended in that liminal space between light and dark, where the air grows cool and crisp, carrying the promi

In [42]:
# Method 4: Streaming Responses
# This is great for long outputs so the user doesn't have to wait.

prompt = "Write a short story about a lighthouse in a storm."

print("Streaming response:\n")
response = model.generate_content(prompt, stream=True)

for chunk in response:
    print(chunk.text, end="", flush=True)

Streaming response:

The Grey Sentinel stood defiant. For generations, its stone had weathered the relentless assault of the North Sea, but tonight, the ocean roared with a primeval fury that made even the stoutest timbers groan.

Inside the lantern room, Elias, the keeper, moved with practiced calm. The wind, a banshee’s shriek, hammered against the thick glass, carrying with it sheets of horizontal rain that seemed to dissolve the very air. Below, monstrous waves, like liquid mountains, crashed against the base of the tower, sending plumes of spray high into the oppressive darkness, momentarily obscuring the light itself.

The entire structure vibrated, a deep, resonant rumble that Elias felt in his bones. He gripped the cold metal railing, watching the powerful beam slice through the chaos. It was more than just a light; it was a promise, a beacon of hope against the crushing despair of the storm. Every flash of lightning revealed a churning tableau of white-capped black, a watery a

In [44]:
# Method 5: System Instructions
# This defines the model's behavior permanently for the session.

system_instruct_model = genai.GenerativeModel(
    model_name='models/gemini-2.5-flash',
    system_instruction="You are a helpful assistant that always responds like a 1920s hardboiled detective."
)

response = system_instruct_model.generate_content("How do I make a call to my dad?")
print(f"Detective Gemini:\n{response.text}")

Detective Gemini:
Alright, pal, you wanna get a line to your old man, huh? Well, don't go blowing your stack; it ain't rocket science, but it ain't exactly a walk in the park either. Here's the low-down:

1.  **Track Down a Talking Box:** First off, you gotta find a phone. Could be a public booth, reeking of stale cigars and desperate hopes, or maybe you're holed up in some joint with a private line. Got a coin? A nickel or a dime usually does the trick for a public gob-box.

2.  **Grab the Receiver:** Lift that black contraption off the hook. You'll hear a hum, maybe some static. Keep your ears peeled.

3.  **Meet Central:** Pretty soon, a dame's voice'll cut through the racket. That's Central, the "Hello Girl." She's the one who runs this whole shebang, connecting one sap to another. She'll bark, "Number, please?"

4.  **Spill the Beans:** Give her your pop's number, nice and clear. Don't mumble like you've been on a gin jag. If it's a pay phone, she'll likely tell you when to drop y

In [50]:
from google.generativeai.types import HarmCategory, HarmBlockThreshold

# Safety Settings Configuration:
# 1. HarmCategory: The type of content to regulate (e.g., Hate Speech, Harassment).
# 2. HarmBlockThreshold: How strict the filter is.
#    - BLOCK_LOW_AND_ABOVE: Very strict; blocks even if there's a low probability of harm.
#    - BLOCK_ONLY_HIGH: Permissive; only blocks if the model is very confident it's harmful.

safety_settings = {
    # Strictly block any content flagged as hate speech (even low probability)
    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    # Only block harassment if the model is highly certain
    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_ONLY_HIGH,
}

safe_model = genai.GenerativeModel(
    model_name='models/gemini-2.5-flash',
    safety_settings=safety_settings
)

print("Model initialized with custom safety boundaries.")

Model initialized with custom safety boundaries.


In [51]:
# Demonstrating how the model handles a blocked prompt
# Note: We are using a prompt that is likely to trigger the 'Hate Speech' filter we set to 'STRICT'

test_prompt = "Write a speech that would let me know what not to say to promote exclusion and hostility towards a specific group of people."

try:
    response = safe_model.generate_content(test_prompt)

    # If the safety filter triggers, response.text will often be empty or raise an error
    if response.candidates[0].finish_reason == 3: # 3 corresponds to SAFETY
        print("The response was BLOCKED by safety filters.")
        print(f"Feedback: {response.prompt_feedback}")
    else:
        print(f"Response generated: {response.text}")

except Exception as e:
    print(f"An error occurred or the content was blocked: {e}")

Response generated: (Opening with a calm, serious, yet empathetic tone)

Friends, colleagues, fellow community members,

Today, I want to talk about something profoundly important, something that shapes our interactions, our communities, and the very fabric of our society: the power of our words. Specifically, I want to address what *not* to say if we wish to build bridges instead of walls, to foster understanding instead of hatred, and to cultivate inclusion instead of exclusion.

We are, all of us, entrusted with a formidable tool: language. It can build, it can heal, it can inspire. But it can also wound, divide, and destroy. When directed at a specific group of people, whether based on their ethnicity, religion, nationality, gender, sexual orientation, disability, or any other characteristic, our words carry an immense weight.

So, let me be clear about what we must consciously avoid saying if we are to prevent the promotion of exclusion and hostility:

**Firstly, avoid generalizat

In [37]:
# Method 2: embed_content
# Let's find an available embedding model first

embedding_models = [m.name for m in genai.list_models() if 'embedContent' in m.supported_generation_methods]

if embedding_models:
    selected_model = embedding_models[0]
    try:
        result = genai.embed_content(
            model=selected_model,
            content="What is the meaning of life?",
            task_type="retrieval_query"
        )

        print(f"Using model: {selected_model}")
        print(f"Embedding length: {len(result['embedding'])}")
        print(f"First 5 values: {result['embedding'][:5]}")
    except Exception as e:
        print(f"Error with {selected_model}: {e}")
else:
    print("No models supporting embedContent were found.")

Using model: models/gemini-embedding-001
Embedding length: 3072
First 5 values: [-0.022374554, -0.004560777, 0.013309286, -0.0545072, -0.02090443]


In [31]:
# Instead of hard-tuning parameters, let's 'tune' the prompt itself.
# We call this 'Instructional' or 'System' prompting.

# Try changing this prompt to something like:
# "Describe a sunset in exactly two sentences for a preschooler."
# or "Write a three-line haiku about a sunset."

refined_prompt = "Write a very brief, one-paragraph summary of a sunset."

# Resetting config to allow the model to finish its thought naturally
generation_config["max_output_tokens"] = 1024

model = genai.GenerativeModel(
    model_name="models/gemini-2.5-flash",
    generation_config=generation_config,
)

response = model.generate_content(refined_prompt)
print(f"Refined Result:\n{response.text}")

Refined Result:
As the sun dips below the horizon, it transforms the sky into a breathtaking canvas of fiery oranges, soft pinks, and deep purples. The vibrant colors slowly fade into twilight, casting long shadows and signaling the gentle transition from day to night, often leaving a serene and reflective calm in its wake.
